# SI Notebook 2: NboDriver resonance diagnostics

This supplementary notebook accompanies the NBO diagnostic section of the article. It demonstrates the direct `NboDriver` API on small organic resonance cases, reports Lewis-score alternatives, and then performs an NRA/NRT-style density fit for benzene.

In [1]:
from pathlib import Path
from contextlib import contextmanager, redirect_stdout, redirect_stderr
import importlib.util
import io
import sys

from veloxchem.molecule import Molecule
from veloxchem.molecularbasis import MolecularBasis
from veloxchem.scfrestdriver import ScfRestrictedDriver
from veloxchem.scfunrestdriver import ScfUnrestrictedDriver

start = Path.cwd().resolve()
repo_root = next(path for path in (start, *start.parents) if (path / 'src' / 'pymodule').exists())
for module_name in ('orbitalanalyzerdriver', 'nbodriver'):
    module_path = repo_root / 'src' / 'pymodule' / f'{module_name}.py'
    spec = importlib.util.spec_from_file_location(f'veloxchem.{module_name}', module_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[f'veloxchem.{module_name}'] = module
    spec.loader.exec_module(module)

from veloxchem.nbodriver import NboComputeOptions, NboDriver

@contextmanager
def quiet_veloxchem():
    with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
        yield

def run_scf(molecule, basis_label='sto-3g', unrestricted=False):
    basis = MolecularBasis.read(molecule, basis_label, ostream=None)
    scf = ScfUnrestrictedDriver() if unrestricted else ScfRestrictedDriver()
    scf.ostream.mute()
    scf.xcfun = 'hf'
    with quiet_veloxchem():
        scf.compute(molecule, basis)
    return basis, scf

def compute_nbo(molecule, basis_label='sto-3g', unrestricted=False, options=None, constraints=None):
    basis, scf = run_scf(molecule, basis_label=basis_label, unrestricted=unrestricted)
    nbo = NboDriver()
    nbo.verbose = False
    results = nbo.compute(molecule, basis, scf.mol_orbs, mode='primary', options=options, constraints=constraints)
    return basis, scf, results

nbo_source = repo_root / 'src' / 'pymodule' / 'nbodriver.py'
print(f'Using local NBO source from {nbo_source}')


Using local NBO source from /home/linares/app/VeloxChem/src/pymodule/nbodriver.py


## Display helpers

The helper functions below turn the raw NBO result dictionaries into compact Lewis-structure tables. When `py3Dmol` is available, `show_nbo_lewis_structures` also displays the selected resonance alternatives with pi bonds, lone pairs, one-electron centers, and positive centers marked on the molecular geometry.

In [2]:
from collections import Counter
from IPython.display import display
import time


def pair_text(pairs):
    return ', '.join(f'{int(i)}-{int(j)}' for i, j in pairs) or 'none'


def alternative_rows(results, limit=8):
    rows = []
    nra_by_rank = {}
    if 'nra' in results:
        nra_by_rank = {
            int(item['rank']): item
            for item in results['nra'].get('structures', [])
        }
    for alternative in results.get('alternatives', [])[:limit]:
        rank = int(alternative.get('rank', 0))
        nra_weight = nra_by_rank.get(rank, {}).get('nra_weight')
        active_counts = Counter(
            (
                candidate.get('type'),
                candidate.get('subtype', ''),
                float(candidate.get('electron_count', 2.0)),
            )
            for candidate in alternative.get('active_nbo_list', [])
        )
        rows.append({
            'rank': rank,
            'score_wt': f"{100.0 * float(alternative.get('weight', 0.0)):.3f}%",
            'nra_wt': 'n/a' if nra_weight is None else f'{100.0 * float(nra_weight):.3f}%',
            'score': f"{float(alternative.get('score', 0.0)):.5f}",
            'e_count': f"{float(alternative.get('selected_lewis_electron_count', 0.0)):.1f}",
            'pi_bonds': pair_text(alternative.get('pi_bonds', [])),
            'one_e': ', '.join(str(atom) for atom in alternative.get('active_one_electron_atoms', [])) or 'none',
            'lp': ', '.join(str(atom) for atom in alternative.get('active_lone_pair_atoms', [])) or 'none',
            'pos': ', '.join(str(atom) for atom in alternative.get('active_positive_atoms', [])) or 'none',
            'active': dict(active_counts),
        })
    return rows


def print_table(rows, columns):
    if not rows:
        print('(no rows)')
        return
    widths = {
        column: max(len(column), *(len(str(row.get(column, ''))) for row in rows))
        for column in columns
    }
    print('  '.join(column.ljust(widths[column]) for column in columns))
    print('  '.join('-' * widths[column] for column in columns))
    for row in rows:
        print('  '.join(str(row.get(column, '')).ljust(widths[column]) for column in columns))


def print_lewis_table(title, results, limit=8):
    print(f'\n{title}')
    print_table(
        alternative_rows(results, limit=limit),
        ['rank', 'score_wt', 'nra_wt', 'score', 'e_count', 'pi_bonds', 'one_e', 'lp', 'pos'],
    )


def show_nbo_lewis_structures(molecule, results, ranks=None, width=360, height=300, max_columns=3):
    try:
        viewer = NboDriver().show_structures(
            results=results,
            molecule=molecule,
            ranks=ranks,
            width=width,
            height=height,
            max_columns=max_columns,
            display=False,
        )
    except ImportError as error:
        print(f'3D Lewis-structure viewer unavailable: {error}')
        return None
    except Exception as error:
        print(f'3D Lewis-structure viewer failed: {error}')
        return None
    display(viewer)
    return viewer


def compute_nbo_timed(molecule, basis_label='sto-3g', unrestricted=False, options=None, constraints=None):
    basis, scf = run_scf(molecule, basis_label=basis_label, unrestricted=unrestricted)
    nbo = NboDriver()
    nbo.verbose = False
    start_time = time.perf_counter()
    results = nbo.compute(
        molecule,
        basis,
        scf.mol_orbs,
        mode='primary',
        options=options,
        constraints=constraints,
    )
    elapsed = time.perf_counter() - start_time
    return basis, scf, nbo, results, elapsed

## Allyl cation, radical, and anion

This section uses the allyl cation, radical, and anion as a compact guide to the `NboDriver` output. The three charge states show how the same carbon framework leads to different Lewis alternatives, occupations, and open-shell bookkeeping. In the radical case, the calculation also illustrates where spin-resolved information enters the NBO analysis. The printed weights are Lewis-score rankings used to compare alternative NBO assignments; they are not VB coefficients and they are not used here as validation criteria.

In [3]:
allyl_xyz = '''
C -1.30000000  0.00000000 0.00000000
C  0.00000000  0.00000000 0.00000000
C  1.30000000  0.00000000 0.00000000
H -1.90000000  0.93000000 0.00000000
H -1.90000000 -0.93000000 0.00000000
H  0.00000000  1.08000000 0.00000000
H  1.90000000  0.93000000 0.00000000
H  1.90000000 -0.93000000 0.00000000
'''

def allyl_molecule(charge, multiplicity):
    molecule = Molecule.read_str(allyl_xyz)
    molecule.set_charge(charge)
    molecule.set_multiplicity(multiplicity)
    return molecule

allyl_pi_structures = {
    'left C1=C2': [(1, 2)],
    'right C2=C3': [(2, 3)],
    'long C1...C3': [(1, 3)],
}
allyl_allowed_pi_bonds = sorted({
    tuple(pair)
    for structure in allyl_pi_structures.values()
    for pair in structure
})

cases = {
    'allyl cation': (allyl_molecule(+1, 1), False),
    'allyl radical': (allyl_molecule(0, 2), True),
    'allyl anion': (allyl_molecule(-1, 1), False),
}

allyl_molecules = {}
allyl_raw_results = {}
allyl_results = {}
for label, (molecule, unrestricted) in cases.items():
    _, _, results = compute_nbo(
        molecule,
        basis_label='sto-3g',
        unrestricted=unrestricted,
        options=NboComputeOptions(max_alternatives=8, lewis_weight_beta=4.0),
        constraints={'allowed_pi_bonds': allyl_allowed_pi_bonds},
    )
    allyl_molecules[label] = molecule
    allyl_raw_results[label] = results
    allyl_results[label] = alternative_rows(results, limit=3)
    print_lewis_table(label, results, limit=3)

allyl_results


allyl cation
rank  score_wt  nra_wt  score     e_count  pi_bonds  one_e  lp    pos
----  --------  ------  --------  -------  --------  -----  ----  ---
1     46.115%   n/a     21.51654  22.0     1-2       none   none  3  
2     46.115%   n/a     21.51654  22.0     2-3       none   none  1  
3     7.771%    n/a     21.07134  22.0     1-3       none   none  2  

allyl radical
rank  score_wt  nra_wt  score     e_count  pi_bonds  one_e  lp    pos 
----  --------  ------  --------  -------  --------  -----  ----  ----
1     50.000%   n/a     22.58636  23.0     1-2       3      none  none
2     50.000%   n/a     22.58636  23.0     2-3       1      none  none
3     0.000%    n/a     0.00000   0.0      1-3       2      none  none

allyl anion
rank  score_wt  nra_wt  score     e_count  pi_bonds  one_e  lp  pos
----  --------  ------  --------  -------  --------  -----  --  ---
1     46.882%   n/a     23.50021  24.0     2-3       none   1   2  
2     46.882%   n/a     23.50021  24.0     1-2   

{'allyl cation': [{'rank': 1,
   'score_wt': '46.115%',
   'nra_wt': 'n/a',
   'score': '21.51654',
   'e_count': '22.0',
   'pi_bonds': '1-2',
   'one_e': 'none',
   'lp': 'none',
   'pos': '3',
   'active': {('BD', 'pi', 2.0): 1}},
  {'rank': 2,
   'score_wt': '46.115%',
   'nra_wt': 'n/a',
   'score': '21.51654',
   'e_count': '22.0',
   'pi_bonds': '2-3',
   'one_e': 'none',
   'lp': 'none',
   'pos': '1',
   'active': {('BD', 'pi', 2.0): 1}},
  {'rank': 3,
   'score_wt': '7.771%',
   'nra_wt': 'n/a',
   'score': '21.07134',
   'e_count': '22.0',
   'pi_bonds': '1-3',
   'one_e': 'none',
   'lp': 'none',
   'pos': '2',
   'active': {('BD', 'pi', 2.0): 1}}],
 'allyl radical': [{'rank': 1,
   'score_wt': '50.000%',
   'nra_wt': 'n/a',
   'score': '22.58636',
   'e_count': '23.0',
   'pi_bonds': '1-2',
   'one_e': '3',
   'lp': 'none',
   'pos': 'none',
   'active': {('SOMO', 'one-electron', 1.0): 1, ('BD', 'pi', 2.0): 1}},
  {'rank': 2,
   'score_wt': '50.000%',
   'nra_wt': 'n/a',
 

## Benzene Kekule and Dewar alternatives

Benzene is used in the article as a symmetry checkpoint. With the two Kekule patterns and three para-Dewar patterns allowed, the expected diagnostic behavior is a degenerate Kekule pair followed by lower-score Dewar alternatives.

In [4]:
benzene = Molecule.read_str('''
C  1.39700000  0.00000000 0.00000000
C  0.69850000  1.20983700 0.00000000
C -0.69850000  1.20983700 0.00000000
C -1.39700000  0.00000000 0.00000000
C -0.69850000 -1.20983700 0.00000000
C  0.69850000 -1.20983700 0.00000000
H  2.48100000  0.00000000 0.00000000
H  1.24050000  2.14883000 0.00000000
H -1.24050000  2.14883000 0.00000000
H -2.48100000  0.00000000 0.00000000
H -1.24050000 -2.14883000 0.00000000
H  1.24050000 -2.14883000 0.00000000
''')
benzene.set_charge(0)
benzene.set_multiplicity(1)

benzene_structures = {
    'Kekule A': [(1, 2), (3, 4), (5, 6)],
    'Kekule B': [(2, 3), (4, 5), (6, 1)],
    'Dewar 1': [(1, 2), (3, 6), (4, 5)],
    'Dewar 2': [(2, 3), (4, 1), (5, 6)],
    'Dewar 3': [(3, 4), (5, 2), (6, 1)],
}
all_benzene_pi_bonds = sorted({
    tuple(pair)
    for structure in benzene_structures.values()
    for pair in structure
})

_, _, benzene_results = compute_nbo(
    benzene,
    basis_label='sto-3g',
    options=NboComputeOptions(max_alternatives=12, lewis_weight_beta=4.0),
    constraints={'allowed_pi_bonds': all_benzene_pi_bonds},
)

print_lewis_table('benzene Kekule and Dewar alternatives', benzene_results, limit=8)
show_nbo_lewis_structures(benzene, benzene_results, ranks=[1, 2, 3, 4, 5, 6], max_columns=3)
alternative_rows(benzene_results, limit=8)


benzene Kekule and Dewar alternatives
rank  score_wt  nra_wt  score     e_count  pi_bonds       one_e  lp    pos 
----  --------  ------  --------  -------  -------------  -----  ----  ----
1     49.995%   n/a     41.04211  42.0     1-6, 2-3, 4-5  none   none  none
2     49.995%   n/a     41.04211  42.0     1-2, 3-4, 5-6  none   none  none
3     0.004%    n/a     38.65877  42.0     1-4, 2-3, 5-6  none   none  none
4     0.004%    n/a     38.65877  42.0     1-2, 3-6, 4-5  none   none  none
5     0.004%    n/a     38.65877  42.0     1-6, 2-5, 3-4  none   none  none
6     0.000%    n/a     33.89211  42.0     1-4, 2-5, 3-6  none   none  none


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

[{'rank': 1,
  'score_wt': '49.995%',
  'nra_wt': 'n/a',
  'score': '41.04211',
  'e_count': '42.0',
  'pi_bonds': '1-6, 2-3, 4-5',
  'one_e': 'none',
  'lp': 'none',
  'pos': 'none',
  'active': {('BD', 'pi', 2.0): 3}},
 {'rank': 2,
  'score_wt': '49.995%',
  'nra_wt': 'n/a',
  'score': '41.04211',
  'e_count': '42.0',
  'pi_bonds': '1-2, 3-4, 5-6',
  'one_e': 'none',
  'lp': 'none',
  'pos': 'none',
  'active': {('BD', 'pi', 2.0): 3}},
 {'rank': 3,
  'score_wt': '0.004%',
  'nra_wt': 'n/a',
  'score': '38.65877',
  'e_count': '42.0',
  'pi_bonds': '1-4, 2-3, 5-6',
  'one_e': 'none',
  'lp': 'none',
  'pos': 'none',
  'active': {('BD', 'pi', 2.0): 3}},
 {'rank': 4,
  'score_wt': '0.004%',
  'nra_wt': 'n/a',
  'score': '38.65877',
  'e_count': '42.0',
  'pi_bonds': '1-2, 3-6, 4-5',
  'one_e': 'none',
  'lp': 'none',
  'pos': 'none',
  'active': {('BD', 'pi', 2.0): 3}},
 {'rank': 5,
  'score_wt': '0.004%',
  'nra_wt': 'n/a',
  'score': '38.65877',
  'e_count': '42.0',
  'pi_bonds': '1-6

## NRA/NRT density-fit layer

The NRA result fits the ab initio NAO density as a convex combination of selected Lewis alternatives. These weights are density-fit weights and should be discussed separately from the Lewis-score weights printed above.

In [5]:
_, _, benzene_nra = compute_nbo(
    benzene,
    basis_label='sto-3g',
    options=NboComputeOptions(
        max_alternatives=12,
        include_nra=True,
        nra_subspace='pi',
        nra_fit_metric='frobenius',
    ),
    constraints={'allowed_pi_bonds': all_benzene_pi_bonds},
)

nra_summary = {
    'relative_residual': benzene_nra['nra']['relative_residual'],
    'relative_total_residual': benzene_nra['nra'].get('relative_total_residual'),
    'structures': [
        {
            'rank': structure['rank'],
            'pi_bonds': structure['pi_bonds'],
            'nra_weight': round(float(structure['nra_weight']), 6),
            'score_weight': round(float(structure['score_weight']), 6),
            'residual_norm': round(float(structure['residual_norm']), 6),
        }
        for structure in benzene_nra['nra']['structures']
    ],
}
nra_summary

{'relative_residual': 0.16763855564870558,
 'relative_total_residual': 0.16763855564870558,
 'structures': [{'rank': 1,
   'pi_bonds': [(1, 6), (2, 3), (4, 5)],
   'nra_weight': 0.333333,
   'score_weight': 0.499946,
   'residual_norm': 2.026431},
  {'rank': 2,
   'pi_bonds': [(1, 2), (3, 4), (5, 6)],
   'nra_weight': 0.333333,
   'score_weight': 0.499946,
   'residual_norm': 2.026431},
  {'rank': 3,
   'pi_bonds': [(1, 4), (2, 3), (5, 6)],
   'nra_weight': 0.111111,
   'score_weight': 3.6e-05,
   'residual_norm': 2.332328},
  {'rank': 4,
   'pi_bonds': [(1, 2), (3, 6), (4, 5)],
   'nra_weight': 0.111111,
   'score_weight': 3.6e-05,
   'residual_norm': 2.332329},
  {'rank': 5,
   'pi_bonds': [(1, 6), (2, 5), (3, 4)],
   'nra_weight': 0.111111,
   'score_weight': 3.6e-05,
   'residual_norm': 2.332329},
  {'rank': 6,
   'pi_bonds': [(1, 4), (2, 5), (3, 6)],
   'nra_weight': 0.0,
   'score_weight': 0.0,
   'residual_norm': 2.847178}]}

## Larger resonance examples from `new_notebook.ipynb`

The article also uses larger conjugated systems to show that the NBO resonance machinery remains readable beyond allyl and benzene. This companion cell brings in naphthalene and anthracene from the acene sequence in `new_notebook.ipynb`, prints the leading Lewis alternatives, and records timing and alternative-count summaries.

In [6]:
larger_examples = {
    'naphthalene': {
        'smiles': 'c1ccc2ccccc2c1',
        'basis_label': 'def2-svp',
        'max_alternatives': 8,
    },
    'anthracene': {
        'smiles': 'c1ccc2cc3ccccc3cc2c1',
        'basis_label': 'def2-svp',
        'max_alternatives': 8,
    },
}

larger_results = {}
larger_summary = []
for label, spec in larger_examples.items():
    molecule = Molecule.read_smiles(spec['smiles'])
    molecule.set_charge(0)
    molecule.set_multiplicity(1)
    _, _, nbo_driver, results, elapsed = compute_nbo_timed(
        molecule,
        basis_label=spec['basis_label'],
        options=NboComputeOptions(
            max_alternatives=spec['max_alternatives'],
            lewis_weight_beta=4.0,
        ),
    )
    larger_results[label] = {
        'molecule': molecule,
        'driver': nbo_driver,
        'results': results,
        'elapsed_s': elapsed,
    }
    larger_summary.append({
        'system': label,
        'basis': spec['basis_label'],
        'atoms': molecule.number_of_atoms(),
        'nbo_candidates': len(results.get('nbo_candidates', [])),
        'alternatives': len(results.get('alternatives', [])),
        'elapsed_s': f'{elapsed:.3f}',
        'primary_counts': results.get('primary', {}).get('counts'),
    })
    print_lewis_table(label, results, limit=8)

larger_summary


naphthalene
rank  score_wt  nra_wt  score     e_count  pi_bonds                  one_e  lp    pos 
----  --------  ------  --------  -------  ------------------------  -----  ----  ----
1     67.309%   n/a     64.43117  68.0     1-10, 2-3, 4-9, 5-6, 7-8  none   none  none
2     16.345%   n/a     64.07733  68.0     1-2, 3-4, 5-6, 7-8, 9-10  none   none  none
3     16.345%   n/a     64.07733  68.0     1-10, 2-3, 4-5, 6-7, 8-9  none   none  none

anthracene
rank  score_wt  nra_wt  score     e_count  pi_bonds                                one_e  lp    pos 
----  --------  ------  --------  -------  --------------------------------------  -----  ----  ----
1     43.530%   n/a     88.85490  94.0     1-14, 2-3, 4-13, 5-6, 7-8, 9-10, 11-12  none   none  none
2     43.530%   n/a     88.85490  94.0     1-14, 2-3, 4-5, 6-11, 7-8, 9-10, 12-13  none   none  none
3     6.470%    n/a     88.37833  94.0     1-2, 3-4, 5-6, 7-8, 9-10, 11-12, 13-14  none   none  none
4     6.470%    n/a     88.37832  9

[{'system': 'naphthalene',
  'basis': 'def2-svp',
  'atoms': 18,
  'nbo_candidates': 188,
  'alternatives': 3,
  'elapsed_s': '0.559',
  'primary_counts': {'CR': 10, 'BD': 24}},
 {'system': 'anthracene',
  'basis': 'def2-svp',
  'atoms': 24,
  'nbo_candidates': 268,
  'alternatives': 4,
  'elapsed_s': '50.589',
  'primary_counts': {'CR': 14, 'BD': 33}}]

## Optional 3D Lewis-structure view

The table output is the portable representation. If `py3Dmol` is installed in the notebook environment, the next cell uses the driver's built-in structure viewer to draw the leading naphthalene alternatives.

In [7]:
naphthalene_entry = larger_results['naphthalene']
show_nbo_lewis_structures(
    naphthalene_entry['molecule'],
    naphthalene_entry['results'],
    ranks=[1, 2, 3],
    max_columns=3,
)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

3Dmol.js failed to load for some reason. Please check your browser console for error messages.